In [3]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
if (ROOT / 'data').exists() is False:
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# === BLOCK 02: HELPERS ===
def _warn(message):
    print(f'WARNING: {message}')

def _to_float(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = value.replace('.', '').replace(',', '.')

    if value in ['', '-', '–', 'nan']:
        return np.nan

    try:
        return float(value)
    except:
        return np.nan

MONTH_NAME_TO_NUM = {
    'januari': 1, 'februari': 2, 'maret': 3, 'april': 4,
    'mei': 5, 'juni': 6, 'juli': 7, 'agustus': 8,
    'september': 9, 'oktober': 10, 'november': 11, 'desember': 12,
}

def _month_name_to_num(month_name):
    if not isinstance(month_name, str):
        return None
    return MONTH_NAME_TO_NUM.get(month_name.strip().lower())

In [5]:
# === BLOCK 03: WFP LOADER ===
def load_wfp_prices(path):
    if not path.exists():
        _warn(f'WFP file not found: {path}')
        return None

    df = pd.read_csv(path, encoding='utf-8-sig')

    if 'admin1' not in df.columns or 'commodity' not in df.columns:
        _warn('Invalid WFP schema')
        return None

    df = df[df['admin1'].astype(str).str.contains('Jawa Timur', case=False, na=False)]
    df = df[df['commodity'].astype(str).str.contains('Rice', case=False, na=False)]

    if df.empty:
        _warn('No WFP rice data for Jawa Timur')
        return None

    df = df[['date', 'price']].copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

    df = df.dropna()

    # Aggregate safely
    df = df.groupby('date')['price'].mean().reset_index()
    df = df.set_index('date').resample('MS')['price'].mean().reset_index()

    df = df.rename(columns={'date': 'ds', 'price': 'y_wfp'})
    df = df.sort_values('ds')

    print(f"WFP loaded: {len(df)} rows")
    return df

In [6]:
# === BLOCK 04: PIHPS LOADER ===
def load_pihps_malang(path):
    if not path.exists():
        _warn(f'File not found: {path}')
        return None

    df_raw = pd.read_excel(path, header=None)
    print("PIHPS raw shape:", df_raw.shape)

    # Detect header row
    header_row = None
    for i in range(len(df_raw)):
        text = ' '.join(df_raw.iloc[i].astype(str))
        if any(k in text for k in ['Komoditas', 'Tanggal', 'No']):
            header_row = i
            break

    if header_row is None:
        header_row = 0

    df = df_raw.iloc[header_row+1:].copy().reset_index(drop=True)

    # Clean header
    header = df_raw.iloc[header_row].astype(str).str.strip()
    header = pd.io.parsers.ParserBase({'names': header})._maybe_dedup_names(header)
    df.columns = header

    df = df.dropna(axis=1, how='all')

    # Detect ID columns
    id_cols = [c for c in df.columns if 'komoditas' in c.lower() or 'no' in c.lower()]
    if not id_cols:
        id_cols = df.columns[:2].tolist()

    # Melt
    long = df.melt(id_vars=id_cols, var_name='ds', value_name='price')

    # Filter rice explicitly
    long = long[long[id_cols[0]].astype(str).str.contains('Beras', case=False, na=False)]

    # Clean date
    long['ds'] = long['ds'].astype(str)
    long = long[long['ds'].str.match(r'\d{1,2}/\d{1,2}/\d{2,4}', na=False)]
    long['ds'] = pd.to_datetime(long['ds'], dayfirst=True, errors='coerce')

    # Clean price
    long['price'] = long['price'].apply(_to_float)
    long = long.dropna(subset=['ds', 'price'])

    df_final = long.groupby('ds')['price'].mean().reset_index()
    df_final = df_final.rename(columns={'price': 'y_pihps'})

    print(f"PIHPS loaded: {len(df_final)} rows")
    return df_final

In [7]:
# === BLOCK 05: BPS PRODUCTION ===
def load_bps_production(paths):
    if not paths:
        _warn('No BPS files found')
        return None

    rows = []

    for path in paths:
        if not path.exists():
            continue

        header_df = pd.read_csv(path, nrows=4, header=None)
        months = header_df.iloc[3].astype(str).tolist()

        df = pd.read_csv(path, skiprows=4, header=None)
        df.columns = months

        df = df.rename(columns={df.columns[0]: 'province'})
        df = df[df['province'].astype(str).str.contains('Jawa Timur', case=False, na=False)]

        if df.empty:
            continue

        year_match = path.name
        year = int(''.join(filter(str.isdigit, year_match))[:4])

        for _, row in df.iterrows():
            for m in months[1:13]:
                month_num = _month_name_to_num(m)
                if month_num is None:
                    continue

                value = _to_float(row.get(m))
                if pd.isna(value):
                    continue

                rows.append({
                    'ds': pd.Timestamp(year=year, month=month_num, day=1),
                    'production_gkg': value
                })

    if not rows:
        return None

    df = pd.DataFrame(rows).sort_values('ds')

    # FIXED: no leakage
    df['production_dev_pct'] = df['production_gkg'].pct_change(12)

    print(f"BPS loaded: {len(df)} rows")
    return df[['ds', 'production_gkg', 'production_dev_pct']]

In [8]:
# === BLOCK 06: BMKG ===
def load_bmkg_rainfall(path):
    if not path.exists():
        _warn(f'BMKG file not found: {path}')
        return None

    df = pd.read_csv(path)

    cols = [str(c) for c in df.columns]

    date_col = next((c for c in cols if 'date' in c.lower() or 'tanggal' in c.lower()), None)
    rain_col = next((c for c in cols if 'rain' in c.lower() or 'curah' in c.lower()), None)

    if not date_col or not rain_col:
        _warn('BMKG columns not detected properly')
        return None

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df[rain_col] = df[rain_col].apply(_to_float)

    df = df.dropna()

    df = df.set_index(date_col).resample('MS')[rain_col].sum().reset_index()
    df = df.rename(columns={date_col: 'ds', rain_col: 'rainfall_mm'})

    df['rainfall_dev_pct'] = df['rainfall_mm'].pct_change(12)

    print(f"BMKG loaded: {len(df)} rows")
    return df

In [9]:
# === BLOCK 07: HARVEST FEATURES ===
def build_harvest_features(date_series):
    ds = pd.to_datetime(date_series)

    HARVEST_MONTHS = [3,4,5,7,8,9]

    months = ds.dt.month

    harvest_window = months.isin(HARVEST_MONTHS).astype(int)
    lean_season = (~months.isin(HARVEST_MONTHS)).astype(int)

    def days_to_next_march(date):
        year = date.year if date.month < 3 else date.year + 1
        return max(0, (pd.Timestamp(year=year, month=3, day=1) - date).days)

    proximity = [
        0 if hw else days_to_next_march(d)
        for d, hw in zip(ds, harvest_window)
    ]

    return pd.DataFrame({
        'ds': ds,
        'harvest_window': harvest_window,
        'lean_season': lean_season,
        'harvest_proximity_days': proximity,
    })

In [10]:
wfp = load_wfp_prices(RAW_DIR / 'wfp_food_prices_idn.csv')
pihps = load_pihps_malang(RAW_DIR / 'Tabel Harga Berdasarkan Daerah.xlsx')
bps = load_bps_production(sorted(RAW_DIR.glob('Produksi Padi Menurut Provinsi (Bulanan)*.csv')))
bmkg = load_bmkg_rainfall(RAW_DIR / 'bmkg_rainfall_jatim.csv')

In [ ]:
# === BLOCK 09: PRICE SELECTION (FIXED) ===
def select_price_series(wfp, pihps):
    if wfp is not None and pihps is not None:
        df = pd.merge(wfp, pihps, on='ds', how='outer')

        # PRIORITY: PIHPS > WFP
        df['y'] = df['y_pihps'].combine_first(df['y_wfp'])

    elif wfp is not None:
        df = wfp.rename(columns={'y_wfp': 'y'})[['ds', 'y']]

    elif pihps is not None:
        df = pihps.rename(columns={'y_pihps': 'y'})[['ds', 'y']]

    else:
        raise ValueError("No price data available")

    df = df.dropna(subset=['y']).sort_values('ds')

    print(f"Final price series: {len(df)} rows")
    return df.reset_index(drop=True)


price_df = select_price_series(wfp, pihps)

ValueError: No price data available

In [ ]:
# === BLOCK 10: MASTER DF (FIXED) ===
def build_master_df(price_df, bps_df, bmkg_df, harvest_df):
    df = price_df.copy()

    if bps_df is not None:
        df = df.merge(bps_df, on='ds', how='left')

    if bmkg_df is not None:
        df = df.merge(bmkg_df, on='ds', how='left')

    df = df.merge(harvest_df, on='ds', how='left')

    # Count missing BEFORE filling
    prod_missing = df['production_dev_pct'].isna().sum() if 'production_dev_pct' in df.columns else len(df)
    rain_missing = df['rainfall_dev_pct'].isna().sum() if 'rainfall_dev_pct' in df.columns else len(df)

    # Safe fill
    if 'production_dev_pct' not in df.columns:
        df['production_dev_pct'] = 0.0
    else:
        df['production_dev_pct'] = df['production_dev_pct'].fillna(0.0)

    if 'rainfall_dev_pct' not in df.columns:
        df['rainfall_dev_pct'] = 0.0
    else:
        df['rainfall_dev_pct'] = df['rainfall_dev_pct'].fillna(0.0)

    # 🔥 IMPORTANT: sort time series
    df = df.sort_values('ds').reset_index(drop=True)

    return df, prod_missing, rain_missing

In [ ]:
# === BLOCK 11: BUILD MASTER DF ===
harvest_df = build_harvest_features(price_df['ds'])

master_df, prod_missing, rain_missing = build_master_df(
    price_df, bps, bmkg, harvest_df
)

# Ensure sorted
master_df = master_df.sort_values('ds')

master_df.to_csv(PROCESSED_DIR / 'master_df.csv', index=False)

In [ ]:
# === BLOCK 11: BUILD MASTER DF ===
harvest_df = build_harvest_features(price_df['ds'])

master_df, prod_missing, rain_missing = build_master_df(
    price_df, bps, bmkg, harvest_df
)

# Ensure sorted
master_df = master_df.sort_values('ds')

master_df.to_csv(PROCESSED_DIR / 'master_df.csv', index=False)

In [ ]:
# === BLOCK 13: TRAIN / HOLDOUT SPLIT ===

# Use time-based split (80%)
split_idx = int(len(master_df) * 0.8)
split_date = master_df.iloc[split_idx]['ds']

train_df = master_df[master_df['ds'] < split_date]
holdout_df = master_df[master_df['ds'] >= split_date]

train_df.to_csv(PROCESSED_DIR / 'train_df.csv', index=False)
holdout_df.to_csv(PROCESSED_DIR / 'holdout_df.csv', index=False)

print(f"Split date: {split_date}")
print(f"Train rows: {len(train_df)}")
print(f"Holdout rows: {len(holdout_df)}")